Classify images into 10 categories:
airplane, car, bird, cat, deer, dog, frog, horse, ship, truck"

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2432s 14us/step


Each image is 32×32×3

32 → width

32 → height

3 → RGB (Red, Green, Blue)

Labels are numbers from 0 to 9

In [3]:
x_train = x_train / 255.0
x_test = x_test / 255.0

Original pixel values: 0 – 255

After dividing: 0 – 1

Helps:

Faster learning

Stable gradients

Better accuracy

👉 CNNs always prefer small, consistent values.

In [4]:
model = models.Sequential([
    
    # Convolution Block 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    #32 filters → model learns 32 different features
    #3×3 filter size → looks at small image regions
    #Slides over image → detects edges, colors, textures
    #Same filter used everywhere → weight sharing
    #ReLU → removes useless negative values
    #This is where CNN actually “sees” the image.

    layers.MaxPooling2D((2,2)),
    #Takes maximum value from each 2×2 region
    #Reduces image size
    #Keeps strongest features
    #Like summarizing important information.
    
    # Convolution Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Flatten and Dense Layers
    layers.Flatten(),
    #Converts feature maps into 1D vector
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])


c:\Users\khana\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [6]:
model.fit(x_train, y_train, epochs=10, batch_size=64)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.4586 - loss: 1.5068
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.5946 - loss: 1.1496
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6447 - loss: 1.0154
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6749 - loss: 0.9284
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6989 - loss: 0.8644
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7197 - loss: 0.8072
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7355 - loss: 0.7551
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7540 - loss: 0.7140
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7670 - loss: 0.6698
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7795 - loss: 0.6348


In [7]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7023 - loss: 0.8992
Test Accuracy: 0.7023000121116638


✅ UPGRADE 1: Make CNN Slightly Deeper

CIFAR-10 images are complex:

colors

textures

overlapping objects

2 Conv layers are not enough to capture this.

So we:

add one more Conv layer

increase feature extraction depth

In [8]:
model = models.Sequential([

    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    # Block 3 (NEW)
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])


✅ UPGRADE 2: Add Batch Normalization 
During training:

feature values keep changing

learning becomes unstable

Batch Normalization:

keeps values in a stable range

allows faster & deeper learning

reduces vanishing/exploding gradients

👉 Almost all modern CNNs use it.

In [9]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])


✅ UPGRADE 3: Add Dropout + Train Longer (Generalization)

CNN is now powerful → risk of overfitting

Dropout:

forces model to not depend on single neurons

improves test accuracy

Also:

CIFAR-10 needs more epochs

In [10]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])


In [11]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(x_test, y_test)
)


Epoch 1/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 27ms/step - accuracy: 0.3807 - loss: 1.6777 - val_accuracy: 0.4772 - val_loss: 1.4447
Epoch 2/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.4999 - loss: 1.3522 - val_accuracy: 0.3719 - val_loss: 2.1929
Epoch 3/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.5466 - loss: 1.2290 - val_accuracy: 0.6105 - val_loss: 1.1685
Epoch 4/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.5811 - loss: 1.1437 - val_accuracy: 0.5688 - val_loss: 1.1724
Epoch 5/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 29ms/step - accuracy: 0.6138 - loss: 1.0615 - val_accuracy: 0.6023 - val_loss: 1.1540
Epoch 6/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.6403 - loss: 0.9984 - val_accuracy: 0.6534 - val_loss: 0.9528
Epoch 7/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.6608 - loss: 0.9436 - val_accuracy: 0.6758 - val_loss: 0.9525
Epoch 8/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.6759 - loss: 0.8928 - 

In [12]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7644 - loss: 0.9399
Test Accuracy: 0.7644000053405762
